# Model Evaluation

In [2]:
# Load the locked dataset and model-selection artifacts

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
)


DATA_PATH = Path(
    "../data/engineered/"
    "ronaldo_1000_ml_engineered.csv"
)

ARTIFACTS_DIR = Path("../artifacts")


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
)

df = (
    df.sort_values("date")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Load persisted contracts
# ---------------------------------------------------------

with open(
    ARTIFACTS_DIR / "feature_spec.json",
    "r",
    encoding="utf-8",
) as f:
    feature_spec = json.load(f)


with open(
    ARTIFACTS_DIR / "validation_policy.json",
    "r",
    encoding="utf-8",
) as f:
    validation_policy = json.load(f)


with open(
    ARTIFACTS_DIR / "preprocessing_spec.json",
    "r",
    encoding="utf-8",
) as f:
    preprocessing_spec = json.load(f)


with open(
    ARTIFACTS_DIR / "baseline_model_spec.json",
    "r",
    encoding="utf-8",
) as f:
    baseline_spec = json.load(f)


with open(
    ARTIFACTS_DIR / "main_model_spec.json",
    "r",
    encoding="utf-8",
) as f:
    main_model_spec = json.load(f)


# ---------------------------------------------------------
# Load validation OOF predictions
# ---------------------------------------------------------

main_oof = pd.read_csv(
    ARTIFACTS_DIR / "main_models_oof.csv",
    parse_dates=["date"],
)

poisson_oof = pd.read_csv(
    ARTIFACTS_DIR / "poisson_baseline_oof.csv",
    parse_dates=["date"],
)


# ---------------------------------------------------------
# Restore core contract
# ---------------------------------------------------------

TARGET = feature_spec[
    "primary_target"
]

SECONDARY_TARGETS = feature_spec[
    "secondary_targets"
]

FULL_FEATURES = feature_spec[
    "full_features"
]

NUMERIC_FEATURES = feature_spec[
    "numeric_features"
]

CATEGORICAL_FEATURES = feature_spec[
    "categorical_features"
]

FEATURE_SETS = preprocessing_spec[
    "feature_sets"
]

FINAL_TEST_START = pd.Timestamp(
    validation_policy[
        "final_test_start"
    ]
)


development_df = df[
    df["date"] < FINAL_TEST_START
].copy()

final_test_df = df[
    df["date"] >= FINAL_TEST_START
].copy()


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(df) == 1333
assert df["match_id"].nunique() == 1333

assert len(main_oof) == 340
assert main_oof["match_id"].nunique() == 340

assert (
    main_oof["date"].max()
    < FINAL_TEST_START
)

assert len(final_test_df) == 78

assert (
    main_model_spec[
        "selection_policy"
    ][
        "final_test_used_for_selection"
    ]
    is False
)

assert (
    main_model_spec[
        "selected_ensemble"
    ]["xgboost_weight"]
    == 0.60
)

assert (
    main_model_spec[
        "selected_ensemble"
    ]["catboost_weight"]
    == 0.40
)


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("PHASE 8 — MODEL EVALUATION")
print("=" * 80)

print(
    "Dataset:",
    df.shape
)

print(
    "Career:",
    df["date"].min().date(),
    "→",
    df["date"].max().date(),
)

print(
    "\nDevelopment pool:",
    len(development_df)
)

print(
    "Final test:",
    len(final_test_df)
)

print(
    "Final test period:",
    final_test_df["date"].min().date(),
    "→",
    final_test_df["date"].max().date(),
)


print("\nVALIDATION OOF")
print("=" * 50)

print(
    "OOF matches:",
    len(main_oof)
)

print(
    "OOF period:",
    main_oof["date"].min().date(),
    "→",
    main_oof["date"].max().date(),
)


print("\nLOCKED MODELS")
print("=" * 50)

print(
    "Ensemble:",
    "60% XGBoost + 40% CatBoost"
)

print(
    "Ensemble macro deviance:",
    main_model_spec[
        "selected_ensemble"
    ][
        "macro_poisson_deviance"
    ]
)

print(
    "XGBoost feature set:",
    main_model_spec[
        "xgboost"
    ]["feature_set"]
)

print(
    "CatBoost feature set:",
    main_model_spec[
        "catboost"
    ]["feature_set"]
)

print(
    "Poisson feature set:",
    main_model_spec[
        "statistical_baseline"
    ]["feature_set"]
)


print("\nTARGETS")
print("=" * 50)

print(
    "Primary:",
    TARGET
)

print(
    "Probability targets:",
    SECONDARY_TARGETS
)


print(
    "\nPhase 8 evaluation contracts: PASSED"
)

PHASE 8 — MODEL EVALUATION
Dataset: (1333, 100)
Career: 2002-08-14 → 2026-08-28

Development pool: 1255
Final test: 78
Final test period: 2025-01-09 → 2026-08-28

VALIDATION OOF
OOF matches: 340
OOF period: 2018-08-18 → 2024-12-06

LOCKED MODELS
Ensemble: 60% XGBoost + 40% CatBoost
Ensemble macro deviance: 1.106496
XGBoost feature set: context
CatBoost feature set: full
Poisson feature set: no_season

TARGETS
Primary: target_goals
Probability targets: ['target_scored', 'target_multi_goal']

Phase 8 evaluation contracts: PASSED


## OOF evaluation

In [5]:
# Evaluate implied scoring probabilities on OOF predictions

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)


# ---------------------------------------------------------
# Join the actual secondary targets onto the OOF predictions
# ---------------------------------------------------------

oof_eval = (
    main_oof
    .merge(
        df[
            [
                "match_id",
                "target_scored",
                "target_multi_goal",
            ]
        ],
        on="match_id",
        how="left",
        validate="one_to_one",
    )
)


assert len(oof_eval) == 340

assert (
    oof_eval[
        [
            "target_scored",
            "target_multi_goal",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


# ---------------------------------------------------------
# Convert expected-goal predictions into Poisson probabilities
# ---------------------------------------------------------

prediction_columns = {
    "XGBoost":
        "xgb_pred",

    "CatBoost":
        "catboost_pred",

    "Ensemble":
        "blend_pred",
}


probability_rows = []


for model_name, pred_col in (
    prediction_columns.items()
):

    lam = np.clip(
        oof_eval[pred_col]
        .to_numpy(dtype=float),
        1e-9,
        None,
    )


    # P(goals >= 1)
    p_scored = (
        1.0
        - np.exp(-lam)
    )


    # P(goals >= 2)
    p_multi = (
        1.0
        - np.exp(-lam)
        * (1.0 + lam)
    )


    y_scored = (
        oof_eval[
            "target_scored"
        ]
        .to_numpy(dtype=int)
    )

    y_multi = (
        oof_eval[
            "target_multi_goal"
        ]
        .to_numpy(dtype=int)
    )


    probability_rows.append({
        "model":
            model_name,

        "scored_brier":
            brier_score_loss(
                y_scored,
                p_scored,
            ),

        "scored_logloss":
            log_loss(
                y_scored,
                p_scored,
            ),

        "scored_auc":
            roc_auc_score(
                y_scored,
                p_scored,
            ),

        "scored_actual_rate":
            y_scored.mean(),

        "scored_predicted_rate":
            p_scored.mean(),

        "multi_brier":
            brier_score_loss(
                y_multi,
                p_multi,
            ),

        "multi_logloss":
            log_loss(
                y_multi,
                p_multi,
            ),

        "multi_auc":
            roc_auc_score(
                y_multi,
                p_multi,
            ),

        "multi_actual_rate":
            y_multi.mean(),

        "multi_predicted_rate":
            p_multi.mean(),
    })


probability_comparison = (
    pd.DataFrame(
        probability_rows
    )
    .sort_values(
        "scored_brier"
    )
    .reset_index(drop=True)
)


print(
    "OOF POISSON-IMPLIED PROBABILITY EVALUATION"
)

print("=" * 140)

print(
    probability_comparison
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Selected ensemble calibration gaps
# ---------------------------------------------------------

ensemble_row = (
    probability_comparison[
        probability_comparison[
            "model"
        ] == "Ensemble"
    ]
    .iloc[0]
)


print("\nSELECTED ENSEMBLE — CALIBRATION SUMMARY")
print("=" * 75)

print(
    "P(score >= 1)"
)

print(
    "Actual rate    :",
    round(
        ensemble_row[
            "scored_actual_rate"
        ],
        6,
    )
)

print(
    "Predicted rate :",
    round(
        ensemble_row[
            "scored_predicted_rate"
        ],
        6,
    )
)

print(
    "Calibration gap:",
    round(
        ensemble_row[
            "scored_predicted_rate"
        ]
        -
        ensemble_row[
            "scored_actual_rate"
        ],
        6,
    )
)


print("\nP(score >= 2)")

print(
    "Actual rate    :",
    round(
        ensemble_row[
            "multi_actual_rate"
        ],
        6,
    )
)

print(
    "Predicted rate :",
    round(
        ensemble_row[
            "multi_predicted_rate"
        ],
        6,
    )
)

print(
    "Calibration gap:",
    round(
        ensemble_row[
            "multi_predicted_rate"
        ]
        -
        ensemble_row[
            "multi_actual_rate"
        ],
        6,
    )
)


print(
    "\nOOF probability evaluation: PASSED"
)

OOF POISSON-IMPLIED PROBABILITY EVALUATION
   model  scored_brier  scored_logloss  scored_auc  scored_actual_rate  scored_predicted_rate  multi_brier  multi_logloss  multi_auc  multi_actual_rate  multi_predicted_rate
CatBoost      0.245825         0.68465    0.567914            0.532353               0.543141     0.146125       0.469024   0.529821           0.176471              0.188743
Ensemble      0.246491         0.68607    0.565378            0.532353               0.534348     0.145167       0.465592   0.532143           0.176471              0.179444
 XGBoost      0.247952         0.68904    0.548768            0.532353               0.527470     0.145107       0.465251   0.527589           0.176471              0.173444

SELECTED ENSEMBLE — CALIBRATION SUMMARY
P(score >= 1)
Actual rate    : 0.532353
Predicted rate : 0.534348
Calibration gap: 0.001995

P(score >= 2)
Actual rate    : 0.176471
Predicted rate : 0.179444
Calibration gap: 0.002973

OOF probability evaluation: PASSED

## Check poisson goal-count distribution calibration

In [8]:
# Poisson distribution implied by the selected ensemble

from math import factorial


y_true = (
    oof_eval["y_true"]
    .to_numpy(dtype=int)
)

lam = np.clip(
    oof_eval["blend_pred"]
    .to_numpy(dtype=float),
    1e-9,
    None,
)


# ---------------------------------------------------------
# Observed distribution
# ---------------------------------------------------------

observed_distribution = {
    "0 goals":
        np.mean(y_true == 0),

    "1 goal":
        np.mean(y_true == 1),

    "2 goals":
        np.mean(y_true == 2),

    "3 goals":
        np.mean(y_true == 3),

    "4+ goals":
        np.mean(y_true >= 4),
}


# ---------------------------------------------------------
# Poisson-implied probabilities for every match
# ---------------------------------------------------------

def poisson_probability(
    lam,
    k,
):
    return (
        np.exp(-lam)
        * np.power(lam, k)
        / factorial(k)
    )


p0 = poisson_probability(
    lam,
    0,
)

p1 = poisson_probability(
    lam,
    1,
)

p2 = poisson_probability(
    lam,
    2,
)

p3 = poisson_probability(
    lam,
    3,
)

p4_plus = (
    1.0
    - p0
    - p1
    - p2
    - p3
)


predicted_distribution = {
    "0 goals":
        p0.mean(),

    "1 goal":
        p1.mean(),

    "2 goals":
        p2.mean(),

    "3 goals":
        p3.mean(),

    "4+ goals":
        p4_plus.mean(),
}


# ---------------------------------------------------------
# Comparison table
# ---------------------------------------------------------

distribution_rows = []


for bucket in observed_distribution:

    observed = (
        observed_distribution[
            bucket
        ]
    )

    predicted = (
        predicted_distribution[
            bucket
        ]
    )


    distribution_rows.append({
        "goal_bucket":
            bucket,

        "observed_rate":
            observed,

        "poisson_predicted_rate":
            predicted,

        "calibration_gap":
            predicted - observed,

        "observed_matches":
            int(
                round(
                    observed
                    * len(y_true)
                )
            ),

        "expected_matches":
            predicted
            * len(y_true),
    })


goal_distribution_comparison = (
    pd.DataFrame(
        distribution_rows
    )
)


print(
    "ENSEMBLE — OOF GOAL-COUNT DISTRIBUTION"
)

print("=" * 100)

print(
    goal_distribution_comparison
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Mean / variance diagnostic
# ---------------------------------------------------------

observed_mean = (
    y_true.mean()
)

observed_variance = (
    y_true.var(ddof=0)
)

dispersion_index = (
    observed_variance
    / observed_mean
)


print("\nOOF COUNT DISPERSION")
print("=" * 60)

print(
    "Observed mean     :",
    round(
        observed_mean,
        6,
    )
)

print(
    "Observed variance :",
    round(
        observed_variance,
        6,
    )
)

print(
    "Variance / mean   :",
    round(
        dispersion_index,
        6,
    )
)

print(
    "Ensemble mean λ   :",
    round(
        lam.mean(),
        6,
    )
)


# ---------------------------------------------------------
# Absolute distribution error
# ---------------------------------------------------------

mean_absolute_distribution_gap = (
    goal_distribution_comparison[
        "calibration_gap"
    ]
    .abs()
    .mean()
)


print(
    "\nMean absolute bucket gap:",
    round(
        mean_absolute_distribution_gap,
        6,
    )
)


print(
    "\nGoal-count distribution check: PASSED"
)

ENSEMBLE — OOF GOAL-COUNT DISTRIBUTION
goal_bucket  observed_rate  poisson_predicted_rate  calibration_gap  observed_matches  expected_matches
    0 goals       0.467647                0.465652        -0.001995               159        158.321847
     1 goal       0.355882                0.354904        -0.000979               121        120.667215
    2 goals       0.132353                0.136250         0.003897                45         46.324926
    3 goals       0.038235                0.035124        -0.003111                13         11.942161
   4+ goals       0.005882                0.008070         0.002188                 2          2.743851

OOF COUNT DISPERSION
Observed mean     : 0.758824
Observed variance : 0.747716
Variance / mean   : 0.985363
Ensemble mean λ   : 0.766466

Mean absolute bucket gap: 0.002434

Goal-count distribution check: PASSED


In [10]:
# Check ensemble calibration across prediction ranges

calibration_df = (
    oof_eval[
        [
            "match_id",
            "date",
            "fold",
            "y_true",
            "blend_pred",
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# Quantile-based calibration groups
# ---------------------------------------------------------

calibration_df[
    "prediction_bin"
] = pd.qcut(
    calibration_df[
        "blend_pred"
    ],
    q=5,
    duplicates="drop",
)


calibration_table = (
    calibration_df
    .groupby(
        "prediction_bin",
        observed=True,
    )
    .agg(
        matches=(
            "match_id",
            "size",
        ),

        min_prediction=(
            "blend_pred",
            "min",
        ),

        max_prediction=(
            "blend_pred",
            "max",
        ),

        mean_prediction=(
            "blend_pred",
            "mean",
        ),

        actual_mean_goals=(
            "y_true",
            "mean",
        ),

        actual_scoring_rate=(
            "y_true",
            lambda x:
                np.mean(x >= 1),
        ),

        actual_multi_goal_rate=(
            "y_true",
            lambda x:
                np.mean(x >= 2),
        ),
    )
    .reset_index()
)


# Poisson-implied probabilities from the
# mean prediction inside each bucket.
calibration_table[
    "implied_scoring_probability"
] = (
    1
    - np.exp(
        -calibration_table[
            "mean_prediction"
        ]
    )
)


calibration_table[
    "implied_multi_goal_probability"
] = (
    1
    -
    np.exp(
        -calibration_table[
            "mean_prediction"
        ]
    )
    *
    (
        1
        +
        calibration_table[
            "mean_prediction"
        ]
    )
)


calibration_table[
    "goal_mean_gap"
] = (
    calibration_table[
        "mean_prediction"
    ]
    -
    calibration_table[
        "actual_mean_goals"
    ]
)


calibration_table[
    "scoring_probability_gap"
] = (
    calibration_table[
        "implied_scoring_probability"
    ]
    -
    calibration_table[
        "actual_scoring_rate"
    ]
)


calibration_table[
    "multi_probability_gap"
] = (
    calibration_table[
        "implied_multi_goal_probability"
    ]
    -
    calibration_table[
        "actual_multi_goal_rate"
    ]
)


print(
    "ENSEMBLE — OOF CALIBRATION BY PREDICTED-RISK QUINTILE"
)

print("=" * 150)

print(
    calibration_table[
        [
            "prediction_bin",
            "matches",
            "min_prediction",
            "max_prediction",
            "mean_prediction",
            "actual_mean_goals",
            "goal_mean_gap",
            "actual_scoring_rate",
            "implied_scoring_probability",
            "scoring_probability_gap",
            "actual_multi_goal_rate",
            "implied_multi_goal_probability",
            "multi_probability_gap",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Check whether higher model predictions correspond
# to higher observed goal rates.
# ---------------------------------------------------------

mean_predictions = (
    calibration_table[
        "mean_prediction"
    ]
    .to_numpy()
)

actual_means = (
    calibration_table[
        "actual_mean_goals"
    ]
    .to_numpy()
)


print("\nRISK-SEPARATION SUMMARY")
print("=" * 70)

print(
    "Lowest quintile predicted λ:",
    round(
        mean_predictions[0],
        6,
    )
)

print(
    "Lowest quintile actual GPG :",
    round(
        actual_means[0],
        6,
    )
)

print(
    "Highest quintile predicted λ:",
    round(
        mean_predictions[-1],
        6,
    )
)

print(
    "Highest quintile actual GPG :",
    round(
        actual_means[-1],
        6,
    )
)

print(
    "Observed GPG spread         :",
    round(
        actual_means[-1]
        - actual_means[0],
        6,
    )
)


is_monotonic = bool(
    np.all(
        np.diff(
            actual_means
        ) >= 0
    )
)


print(
    "Observed GPG monotonic      :",
    is_monotonic
)


print(
    "\nRisk-bucket calibration check: PASSED"
)

ENSEMBLE — OOF CALIBRATION BY PREDICTED-RISK QUINTILE
prediction_bin  matches  min_prediction  max_prediction  mean_prediction  actual_mean_goals  goal_mean_gap  actual_scoring_rate  implied_scoring_probability  scoring_probability_gap  actual_multi_goal_rate  implied_multi_goal_probability  multi_probability_gap
(0.598, 0.713]       68        0.599467        0.712879         0.675080           0.750000      -0.074920             0.529412                     0.490884                -0.038528                0.161765                        0.147190              -0.014574
(0.713, 0.741]       68        0.712934        0.740581         0.728042           0.661765       0.066277             0.455882                     0.517147                 0.061264                0.176471                        0.165609              -0.010862
 (0.741, 0.79]       68        0.741810        0.789647         0.762806           0.647059       0.115747             0.455882                     0.533644       

## Fit locked models on development data and evaluate final test

In [13]:
# Fit the LOCKED XGBoost and CatBoost configurations
# on all pre-2025 development data, then evaluate once
# on the 2025-2026 final test.

import xgboost as xgb

from catboost import CatBoostRegressor

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


# ---------------------------------------------------------
# Locked model specifications
# ---------------------------------------------------------

XGB_FEATURES = FEATURE_SETS[
    main_model_spec[
        "xgboost"
    ]["feature_set"]
]

CATBOOST_FEATURES = FEATURE_SETS[
    main_model_spec[
        "catboost"
    ]["feature_set"]
]


XGB_PARAMS = (
    main_model_spec[
        "xgboost"
    ]["params"]
    .copy()
)

CATBOOST_PARAMS = (
    main_model_spec[
        "catboost"
    ]["params"]
    .copy()
)


# n_jobs was used during development but was not
# required in the persisted specification.
XGB_PARAMS["n_jobs"] = -1


# ---------------------------------------------------------
# XGBoost preprocessing
# ---------------------------------------------------------

def build_locked_xgb_preprocessor(
    features
):

    numeric = [
        feature
        for feature in features
        if feature in NUMERIC_FEATURES
    ]

    categorical = [
        feature
        for feature in features
        if feature in CATEGORICAL_FEATURES
    ]

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                numeric,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
                categorical,
            ),
        ],
        remainder="drop",
    )


# ---------------------------------------------------------
# Prepare development and final-test data
# ---------------------------------------------------------

y_dev = (
    development_df[TARGET]
    .to_numpy(dtype=float)
)

y_test = (
    final_test_df[TARGET]
    .to_numpy(dtype=float)
)


# ---------------------------------------------------------
# Fit locked XGBoost
# ---------------------------------------------------------

locked_xgb = Pipeline(
    steps=[
        (
            "preprocessor",
            build_locked_xgb_preprocessor(
                XGB_FEATURES
            ),
        ),
        (
            "model",
            xgb.XGBRegressor(
                **XGB_PARAMS
            ),
        ),
    ]
)


locked_xgb.fit(
    development_df[
        XGB_FEATURES
    ],
    y_dev,
)


xgb_test_pred = (
    locked_xgb.predict(
        final_test_df[
            XGB_FEATURES
        ]
    )
)

xgb_test_pred = np.clip(
    xgb_test_pred,
    1e-9,
    None,
)


# ---------------------------------------------------------
# Fit locked CatBoost
# ---------------------------------------------------------

X_dev_cat = (
    development_df[
        CATBOOST_FEATURES
    ]
    .copy()
)

X_test_cat = (
    final_test_df[
        CATBOOST_FEATURES
    ]
    .copy()
)


cat_features = [
    feature
    for feature in CATBOOST_FEATURES
    if feature in CATEGORICAL_FEATURES
]


for column in cat_features:

    X_dev_cat[column] = (
        X_dev_cat[column]
        .fillna("__MISSING__")
        .astype(str)
    )

    X_test_cat[column] = (
        X_test_cat[column]
        .fillna("__MISSING__")
        .astype(str)
    )


locked_catboost = (
    CatBoostRegressor(
        **CATBOOST_PARAMS
    )
)


locked_catboost.fit(
    X_dev_cat,
    y_dev,
    cat_features=cat_features,
)


catboost_test_pred = (
    locked_catboost.predict(
        X_test_cat
    )
)

catboost_test_pred = np.clip(
    catboost_test_pred,
    1e-9,
    None,
)


# ---------------------------------------------------------
# Locked 60/40 ensemble
# ---------------------------------------------------------

XGB_WEIGHT = (
    main_model_spec[
        "selected_ensemble"
    ]["xgboost_weight"]
)

CATBOOST_WEIGHT = (
    main_model_spec[
        "selected_ensemble"
    ]["catboost_weight"]
)


ensemble_test_pred = (
    XGB_WEIGHT
    * xgb_test_pred
    +
    CATBOOST_WEIGHT
    * catboost_test_pred
)


# ---------------------------------------------------------
# Evaluation helper
# ---------------------------------------------------------

def evaluate_final_predictions(
    name,
    y_true,
    y_pred,
):

    return {
        "model":
            name,

        "poisson_deviance":
            mean_poisson_deviance(
                y_true,
                y_pred,
            ),

        "mae":
            mean_absolute_error(
                y_true,
                y_pred,
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred,
                )
            ),

        "actual_mean":
            y_true.mean(),

        "predicted_mean":
            y_pred.mean(),

        "mean_bias":
            y_pred.mean()
            - y_true.mean(),
    }


final_test_results = pd.DataFrame([
    evaluate_final_predictions(
        "XGBoost",
        y_test,
        xgb_test_pred,
    ),

    evaluate_final_predictions(
        "CatBoost",
        y_test,
        catboost_test_pred,
    ),

    evaluate_final_predictions(
        "Ensemble",
        y_test,
        ensemble_test_pred,
    ),
])


final_test_results = (
    final_test_results
    .sort_values(
        "poisson_deviance"
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# Save individual match predictions immediately
# ---------------------------------------------------------

final_test_predictions = pd.DataFrame({
    "match_id":
        final_test_df[
            "match_id"
        ].values,

    "date":
        final_test_df[
            "date"
        ].values,

    "team":
        final_test_df[
            "team"
        ].values,

    "opponent":
        final_test_df[
            "opponent"
        ].values,

    "competition":
        final_test_df[
            "competition"
        ].values,

    "actual_goals":
        y_test.astype(int),

    "xgb_pred":
        xgb_test_pred,

    "catboost_pred":
        catboost_test_pred,

    "ensemble_pred":
        ensemble_test_pred,
})


FINAL_TEST_PREDICTIONS_PATH = (
    ARTIFACTS_DIR
    / "final_test_predictions.csv"
)

final_test_predictions.to_csv(
    FINAL_TEST_PREDICTIONS_PATH,
    index=False,
)


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(
    final_test_predictions
) == 78

assert (
    final_test_predictions[
        "match_id"
    ].nunique()
    == 78
)

assert np.isfinite(
    final_test_predictions[
        [
            "xgb_pred",
            "catboost_pred",
            "ensemble_pred",
        ]
    ].to_numpy()
).all()


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print(
    "FINAL UNTOUCHED TEST — 2025-2026"
)

print("=" * 100)

print(
    "Matches:",
    len(final_test_df)
)

print(
    "Period:",
    final_test_df[
        "date"
    ].min().date(),
    "→",
    final_test_df[
        "date"
    ].max().date(),
)

print(
    "Actual goals:",
    int(
        y_test.sum()
    )
)


print("\nFINAL TEST RESULTS")
print("=" * 100)

print(
    final_test_results
    .round(6)
    .to_string(index=False)
)


print("\nPREDICTION RANGE")
print("=" * 70)

for name, pred in [
    (
        "XGBoost",
        xgb_test_pred,
    ),
    (
        "CatBoost",
        catboost_test_pred,
    ),
    (
        "Ensemble",
        ensemble_test_pred,
    ),
]:

    print(
        f"{name:<10}",
        f"min={pred.min():.4f}",
        f"median={np.median(pred):.4f}",
        f"max={pred.max():.4f}",
    )


print(
    "\nSaved:",
    FINAL_TEST_PREDICTIONS_PATH
)

print(
    "\nFINAL TEST EVALUATION: COMPLETE"
)

FINAL UNTOUCHED TEST — 2025-2026
Matches: 78
Period: 2025-01-09 → 2026-08-28
Actual goals: 62

FINAL TEST RESULTS
   model  poisson_deviance      mae     rmse  actual_mean  predicted_mean  mean_bias
CatBoost          0.794883 0.546099 0.685570     0.794872        0.866915   0.072044
Ensemble          0.803222 0.575220 0.691029     0.794872        0.806061   0.011189
 XGBoost          0.817682 0.594634 0.699497     0.794872        0.765492  -0.029380

PREDICTION RANGE
XGBoost    min=0.6839 median=0.7731 max=0.7920
CatBoost   min=0.5741 median=0.9002 max=0.9946
Ensemble   min=0.6434 median=0.8218 max=0.8730

Saved: ..\artifacts\final_test_predictions.csv

FINAL TEST EVALUATION: COMPLETE


In [15]:
# Final-test probability and count-distribution evaluation

from math import factorial

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)


# ---------------------------------------------------------
# Actual binary outcomes
# ---------------------------------------------------------

y_scored_test = (
    final_test_df[
        "target_scored"
    ]
    .to_numpy(dtype=int)
)

y_multi_test = (
    final_test_df[
        "target_multi_goal"
    ]
    .to_numpy(dtype=int)
)


# ---------------------------------------------------------
# Selected ensemble expected goals
# ---------------------------------------------------------

lam = np.clip(
    ensemble_test_pred,
    1e-9,
    None,
)


# Poisson-implied probabilities
p_scored = (
    1.0
    - np.exp(-lam)
)

p_multi = (
    1.0
    - np.exp(-lam)
    * (1.0 + lam)
)


# ---------------------------------------------------------
# Probability metrics
# ---------------------------------------------------------

probability_metrics = {
    "scored_brier":
        brier_score_loss(
            y_scored_test,
            p_scored,
        ),

    "scored_logloss":
        log_loss(
            y_scored_test,
            p_scored,
        ),

    "scored_auc":
        roc_auc_score(
            y_scored_test,
            p_scored,
        ),

    "scored_actual_rate":
        y_scored_test.mean(),

    "scored_predicted_rate":
        p_scored.mean(),

    "multi_brier":
        brier_score_loss(
            y_multi_test,
            p_multi,
        ),

    "multi_logloss":
        log_loss(
            y_multi_test,
            p_multi,
        ),

    "multi_auc":
        roc_auc_score(
            y_multi_test,
            p_multi,
        ),

    "multi_actual_rate":
        y_multi_test.mean(),

    "multi_predicted_rate":
        p_multi.mean(),
}


print(
    "FINAL TEST — ENSEMBLE PROBABILITY EVALUATION"
)

print("=" * 85)

for metric, value in (
    probability_metrics.items()
):

    print(
        f"{metric:<28}: "
        f"{value:.6f}"
    )


# ---------------------------------------------------------
# Goal-count distribution
# ---------------------------------------------------------

def poisson_probability(
    lam,
    k,
):
    return (
        np.exp(-lam)
        * np.power(lam, k)
        / factorial(k)
    )


p0 = poisson_probability(lam, 0)
p1 = poisson_probability(lam, 1)
p2 = poisson_probability(lam, 2)
p3 = poisson_probability(lam, 3)

p4_plus = (
    1.0
    - p0
    - p1
    - p2
    - p3
)


observed_rates = {
    "0 goals":
        np.mean(y_test == 0),

    "1 goal":
        np.mean(y_test == 1),

    "2 goals":
        np.mean(y_test == 2),

    "3 goals":
        np.mean(y_test == 3),

    "4+ goals":
        np.mean(y_test >= 4),
}


predicted_rates = {
    "0 goals":
        p0.mean(),

    "1 goal":
        p1.mean(),

    "2 goals":
        p2.mean(),

    "3 goals":
        p3.mean(),

    "4+ goals":
        p4_plus.mean(),
}


distribution_rows = []


for bucket in observed_rates:

    observed = (
        observed_rates[
            bucket
        ]
    )

    predicted = (
        predicted_rates[
            bucket
        ]
    )

    distribution_rows.append({
        "goal_bucket":
            bucket,

        "observed_rate":
            observed,

        "poisson_predicted_rate":
            predicted,

        "calibration_gap":
            predicted - observed,

        "observed_matches":
            int(
                np.sum(
                    (
                        y_test == int(
                            bucket[0]
                        )
                    )
                    if bucket != "4+ goals"
                    else
                    (
                        y_test >= 4
                    )
                )
            ),

        "expected_matches":
            predicted
            * len(y_test),
    })


final_distribution = pd.DataFrame(
    distribution_rows
)


print(
    "\nFINAL TEST — GOAL-COUNT DISTRIBUTION"
)

print("=" * 100)

print(
    final_distribution
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Dispersion
# ---------------------------------------------------------

observed_mean = (
    y_test.mean()
)

observed_variance = (
    y_test.var(ddof=0)
)

dispersion = (
    observed_variance
    / observed_mean
)


mean_bucket_gap = (
    final_distribution[
        "calibration_gap"
    ]
    .abs()
    .mean()
)


print(
    "\nFINAL TEST — DISTRIBUTION SUMMARY"
)

print("=" * 70)

print(
    "Observed mean              :",
    round(
        observed_mean,
        6,
    )
)

print(
    "Observed variance          :",
    round(
        observed_variance,
        6,
    )
)

print(
    "Variance / mean            :",
    round(
        dispersion,
        6,
    )
)

print(
    "Ensemble mean lambda       :",
    round(
        lam.mean(),
        6,
    )
)

print(
    "Mean absolute bucket gap   :",
    round(
        mean_bucket_gap,
        6,
    )
)


print(
    "\nFinal-test probability/distribution evaluation: COMPLETE"
)

FINAL TEST — ENSEMBLE PROBABILITY EVALUATION
scored_brier                : 0.234051
scored_logloss              : 0.661025
scored_auc                  : 0.652357
scored_actual_rate          : 0.628205
scored_predicted_rate       : 0.552678
multi_brier                 : 0.136740
multi_logloss               : 0.443402
multi_auc                   : 0.684024
multi_actual_rate           : 0.166667
multi_predicted_rate        : 0.193540

FINAL TEST — GOAL-COUNT DISTRIBUTION
goal_bucket  observed_rate  poisson_predicted_rate  calibration_gap  observed_matches  expected_matches
    0 goals       0.371795                0.447322         0.075527                29         34.891097
     1 goal       0.461538                0.359138        -0.102401                36         28.012756
    2 goals       0.166667                0.144905        -0.021762                13         11.302601
    3 goals       0.000000                0.039160         0.039160                 0          3.054462
   4+ g

## Diagnose recent goal-count dispersion

In [18]:
# Diagnose whether underdispersion emerged before the final test

dispersion_df = (
    df[
        [
            "date",
            "target_goals",
        ]
    ]
    .copy()
)

dispersion_df["year"] = (
    dispersion_df["date"].dt.year
)


def summarize_count_distribution(
    data,
    label,
):

    goals = (
        data["target_goals"]
        .to_numpy(dtype=float)
    )

    mean_goals = goals.mean()

    variance = goals.var(
        ddof=0
    )

    dispersion = (
        variance / mean_goals
        if mean_goals > 0
        else np.nan
    )

    return {
        "period":
            label,

        "matches":
            len(goals),

        "goals":
            int(goals.sum()),

        "mean":
            mean_goals,

        "variance":
            variance,

        "dispersion":
            dispersion,

        "zero_rate":
            np.mean(goals == 0),

        "one_goal_rate":
            np.mean(goals == 1),

        "two_plus_rate":
            np.mean(goals >= 2),

        "three_plus_rate":
            np.mean(goals >= 3),
    }


# ---------------------------------------------------------
# Predefined historical periods
# ---------------------------------------------------------

periods = [
    (
        "Full career",
        df,
    ),

    (
        "Pre-2018",
        df[
            df["date"]
            < "2018-01-01"
        ],
    ),

    (
        "2018-2020",
        df[
            (df["date"] >= "2018-01-01")
            &
            (df["date"] < "2021-01-01")
        ],
    ),

    (
        "2021-2022",
        df[
            (df["date"] >= "2021-01-01")
            &
            (df["date"] < "2023-01-01")
        ],
    ),

    (
        "2023-2024",
        df[
            (df["date"] >= "2023-01-01")
            &
            (df["date"] < "2025-01-01")
        ],
    ),

    (
        "2025-2026 FINAL TEST",
        df[
            df["date"]
            >= FINAL_TEST_START
        ],
    ),
]


period_distribution = pd.DataFrame(
    [
        summarize_count_distribution(
            period_data,
            label,
        )
        for label, period_data
        in periods
    ]
)


print(
    "GOAL-COUNT DISPERSION BY CAREER PERIOD"
)

print("=" * 125)

print(
    period_distribution
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Rolling recent windows ending BEFORE the final test
# ---------------------------------------------------------

pre_test_df = (
    df[
        df["date"]
        < FINAL_TEST_START
    ]
    .copy()
)


rolling_rows = []


for window in [
    50,
    100,
    150,
    200,
    300,
]:

    recent = (
        pre_test_df
        .tail(window)
    )

    summary = (
        summarize_count_distribution(
            recent,
            f"Last {window} pre-test matches",
        )
    )

    summary[
        "start_date"
    ] = recent[
        "date"
    ].min().date()

    summary[
        "end_date"
    ] = recent[
        "date"
    ].max().date()

    rolling_rows.append(
        summary
    )


rolling_distribution = pd.DataFrame(
    rolling_rows
)


print(
    "\nRECENT WINDOWS — USING ONLY PRE-2025 DATA"
)

print("=" * 135)

print(
    rolling_distribution[
        [
            "period",
            "start_date",
            "end_date",
            "matches",
            "goals",
            "mean",
            "variance",
            "dispersion",
            "zero_rate",
            "one_goal_rate",
            "two_plus_rate",
            "three_plus_rate",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Calendar-year diagnostic
# ---------------------------------------------------------

year_rows = []


for year, year_df in (
    dispersion_df
    .groupby("year")
):

    # Recent years only, and keep the final-test years
    # visible for comparison rather than model fitting.
    if year >= 2018:

        year_rows.append(
            summarize_count_distribution(
                year_df,
                str(year),
            )
        )


year_distribution = pd.DataFrame(
    year_rows
)


print(
    "\nRECENT CALENDAR-YEAR DISPERSION"
)

print("=" * 120)

print(
    year_distribution
    .round(6)
    .to_string(index=False)
)


print(
    "\nRecent-dispersion diagnostic: COMPLETE"
)

GOAL-COUNT DISPERSION BY CAREER PERIOD
              period  matches  goals     mean  variance  dispersion  zero_rate  one_goal_rate  two_plus_rate  three_plus_rate
         Full career     1333    978 0.733683  0.818048    1.114987   0.507127       0.311328       0.181545         0.049512
            Pre-2018      886    624 0.704289  0.853864    1.212378   0.539503       0.282167       0.178330         0.054176
           2018-2020      148    132 0.891892  0.839664    0.941441   0.398649       0.378378       0.222973         0.054054
           2021-2022      111     63 0.567568  0.659849    1.162591   0.603604       0.261261       0.135135         0.036036
           2023-2024      110     97 0.881818  0.795124    0.901687   0.390909       0.400000       0.209091         0.054545
2025-2026 FINAL TEST       78     62 0.794872  0.496384    0.624483   0.371795       0.461538       0.166667         0.000000

RECENT WINDOWS — USING ONLY PRE-2025 DATA
                   period start_date

In [20]:
# Estimate dispersion relative to the ensemble's
# match-specific expected-goal predictions using OOF data only.

dispersion_eval = (
    oof_eval[
        [
            "match_id",
            "date",
            "fold",
            "y_true",
            "blend_pred",
        ]
    ]
    .copy()
)


y = (
    dispersion_eval[
        "y_true"
    ]
    .to_numpy(dtype=float)
)

mu = np.clip(
    dispersion_eval[
        "blend_pred"
    ]
    .to_numpy(dtype=float),
    1e-9,
    None,
)


# ---------------------------------------------------------
# Pearson dispersion
#
# For a correctly specified Poisson model:
#     Var(Y | X) = mu
#
# Pearson dispersion should therefore be near 1.
# < 1 suggests conditional underdispersion.
# > 1 suggests conditional overdispersion.
# ---------------------------------------------------------

pearson_components = (
    np.square(
        y - mu
    )
    / mu
)

overall_pearson_dispersion = (
    pearson_components.mean()
)


# ---------------------------------------------------------
# Fold-level conditional dispersion
# ---------------------------------------------------------

fold_rows = []


for fold, fold_df in (
    dispersion_eval
    .groupby("fold")
):

    fold_y = (
        fold_df[
            "y_true"
        ]
        .to_numpy(dtype=float)
    )

    fold_mu = np.clip(
        fold_df[
            "blend_pred"
        ]
        .to_numpy(dtype=float),
        1e-9,
        None,
    )

    pearson = (
        np.square(
            fold_y - fold_mu
        )
        / fold_mu
    )

    fold_rows.append({
        "fold":
            fold,

        "matches":
            len(fold_df),

        "actual_mean":
            fold_y.mean(),

        "predicted_mean":
            fold_mu.mean(),

        "raw_variance":
            fold_y.var(ddof=0),

        "raw_dispersion":
            (
                fold_y.var(ddof=0)
                / fold_y.mean()
            ),

        "pearson_dispersion":
            pearson.mean(),

        "mean_pearson_residual":
            np.mean(
                (
                    fold_y - fold_mu
                )
                / np.sqrt(fold_mu)
            ),
    })


fold_dispersion = pd.DataFrame(
    fold_rows
)


print(
    "OOF CONDITIONAL DISPERSION — SELECTED ENSEMBLE"
)

print("=" * 110)

print(
    fold_dispersion
    .round(6)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Recent OOF periods
# ---------------------------------------------------------

recent_periods = [
    (
        "All OOF",
        dispersion_eval,
    ),

    (
        "2021-2024 OOF",
        dispersion_eval[
            dispersion_eval[
                "date"
            ] >= "2021-01-01"
        ],
    ),

    (
        "2023-2024 OOF",
        dispersion_eval[
            dispersion_eval[
                "date"
            ] >= "2023-01-01"
        ],
    ),

    (
        "2024 OOF",
        dispersion_eval[
            dispersion_eval[
                "date"
            ] >= "2024-01-01"
        ],
    ),
]


recent_rows = []


for label, period_df in recent_periods:

    period_y = (
        period_df[
            "y_true"
        ]
        .to_numpy(dtype=float)
    )

    period_mu = np.clip(
        period_df[
            "blend_pred"
        ]
        .to_numpy(dtype=float),
        1e-9,
        None,
    )

    pearson = (
        np.square(
            period_y - period_mu
        )
        / period_mu
    )

    recent_rows.append({
        "period":
            label,

        "matches":
            len(period_df),

        "actual_mean":
            period_y.mean(),

        "predicted_mean":
            period_mu.mean(),

        "raw_dispersion":
            (
                period_y.var(ddof=0)
                / period_y.mean()
            ),

        "pearson_dispersion":
            pearson.mean(),
    })


recent_conditional_dispersion = (
    pd.DataFrame(
        recent_rows
    )
)


print(
    "\nRECENT OOF CONDITIONAL DISPERSION"
)

print("=" * 100)

print(
    recent_conditional_dispersion
    .round(6)
    .to_string(index=False)
)


print(
    "\nOverall OOF Pearson dispersion:",
    round(
        overall_pearson_dispersion,
        6,
    )
)


print(
    "\nConditional-dispersion diagnostic: COMPLETE"
)

OOF CONDITIONAL DISPERSION — SELECTED ENSEMBLE
  fold  matches  actual_mean  predicted_mean  raw_variance  raw_dispersion  pearson_dispersion  mean_pearson_residual
Fold 1      158     0.791139        0.793762      0.722200        0.912861            0.904759              -0.004185
Fold 2       72     0.500000        0.738552      0.638889        1.277778            0.921106              -0.279602
Fold 3       59     0.915254        0.690635      0.857225        0.936598            1.348377               0.270639
Fold 4       51     0.843137        0.809033      0.720492        0.854537            0.864414               0.034938

RECENT OOF CONDITIONAL DISPERSION
       period  matches  actual_mean  predicted_mean  raw_dispersion  pearson_dispersion
      All OOF      340     0.758824        0.766466        0.985363            0.979150
2021-2024 OOF      221     0.723982        0.757547        1.038518            1.007713
2023-2024 OOF      110     0.881818        0.745529        0.901

In [22]:
# Persist Phase 8 evaluation results and conclusions

EVALUATION_SPEC_PATH = (
    ARTIFACTS_DIR
    / "evaluation_spec.json"
)


# ---------------------------------------------------------
# Extract final-test model metrics
# ---------------------------------------------------------

final_metrics = {}


for _, row in (
    final_test_results
    .iterrows()
):

    model_name = (
        str(row["model"])
        .lower()
    )

    final_metrics[
        model_name
    ] = {
        "poisson_deviance":
            float(
                row[
                    "poisson_deviance"
                ]
            ),

        "mae":
            float(
                row[
                    "mae"
                ]
            ),

        "rmse":
            float(
                row[
                    "rmse"
                ]
            ),

        "actual_mean":
            float(
                row[
                    "actual_mean"
                ]
            ),

        "predicted_mean":
            float(
                row[
                    "predicted_mean"
                ]
            ),

        "mean_bias":
            float(
                row[
                    "mean_bias"
                ]
            ),
    }


# ---------------------------------------------------------
# Final evaluation specification
# ---------------------------------------------------------

evaluation_spec = {
    "phase":
        "Phase 8 - Model Evaluation",

    "final_test": {
        "start":
            str(
                final_test_df[
                    "date"
                ].min().date()
            ),

        "end":
            str(
                final_test_df[
                    "date"
                ].max().date()
            ),

        "matches":
            int(
                len(
                    final_test_df
                )
            ),

        "actual_goals":
            int(
                y_test.sum()
            ),

        "actual_gpg":
            float(
                y_test.mean()
            ),

        "models":
            final_metrics,
    },


    "selected_model": {
        "name":
            "60% XGBoost + 40% CatBoost ensemble",

        "selection_source":
            "pre-2025 temporal validation",

        "changed_after_final_test":
            False,

        "final_test_poisson_deviance":
            float(
                final_metrics[
                    "ensemble"
                ][
                    "poisson_deviance"
                ]
            ),

        "final_test_mae":
            float(
                final_metrics[
                    "ensemble"
                ][
                    "mae"
                ]
            ),

        "final_test_rmse":
            float(
                final_metrics[
                    "ensemble"
                ][
                    "rmse"
                ]
            ),

        "final_test_mean_bias":
            float(
                final_metrics[
                    "ensemble"
                ][
                    "mean_bias"
                ]
            ),
    },


    "probability_evaluation": {
        "final_test_scored_brier":
            float(
                probability_metrics[
                    "scored_brier"
                ]
            ),

        "final_test_scored_logloss":
            float(
                probability_metrics[
                    "scored_logloss"
                ]
            ),

        "final_test_scored_auc":
            float(
                probability_metrics[
                    "scored_auc"
                ]
            ),

        "final_test_scored_actual_rate":
            float(
                probability_metrics[
                    "scored_actual_rate"
                ]
            ),

        "final_test_scored_predicted_rate":
            float(
                probability_metrics[
                    "scored_predicted_rate"
                ]
            ),

        "final_test_multi_brier":
            float(
                probability_metrics[
                    "multi_brier"
                ]
            ),

        "final_test_multi_logloss":
            float(
                probability_metrics[
                    "multi_logloss"
                ]
            ),

        "final_test_multi_auc":
            float(
                probability_metrics[
                    "multi_auc"
                ]
            ),

        "final_test_multi_actual_rate":
            float(
                probability_metrics[
                    "multi_actual_rate"
                ]
            ),

        "final_test_multi_predicted_rate":
            float(
                probability_metrics[
                    "multi_predicted_rate"
                ]
            ),
    },


    "distribution_evaluation": {
        "oof_pearson_dispersion":
            float(
                overall_pearson_dispersion
            ),

        "final_test_raw_dispersion":
            float(
                dispersion
            ),

        "final_test_mean_absolute_bucket_gap":
            float(
                mean_bucket_gap
            ),

        "simulation_distribution":
            "Poisson",

        "distribution_decision":
            (
                "Retain Poisson as the primary "
                "simulation distribution."
            ),

        "decision_basis": [
            (
                "Overall OOF conditional Pearson "
                "dispersion was approximately 1."
            ),

            (
                "2021-2024 OOF conditional dispersion "
                "was approximately 1."
            ),

            (
                "Recent underdispersion was not stable "
                "across pre-test validation periods."
            ),

            (
                "The 2025-2026 final test was strongly "
                "underdispersed, but was not used to "
                "retune the simulation distribution."
            ),
        ],
    },


    "limitations": [
        (
            "Individual-match discrimination is "
            "moderate despite strong aggregate "
            "expected-goal calibration."
        ),

        (
            "The model can miss abrupt scoring-regime "
            "changes, as seen across temporal folds."
        ),

        (
            "The final-test goal-count distribution "
            "was substantially more underdispersed "
            "than Poisson."
        ),

        (
            "Future milestone forecasts should report "
            "uncertainty rather than a single certain "
            "1000th-goal date."
        ),
    ],
}


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

with open(
    EVALUATION_SPEC_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        evaluation_spec,
        f,
        indent=2,
    )


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert (
    evaluation_spec[
        "selected_model"
    ][
        "changed_after_final_test"
    ]
    is False
)

assert (
    evaluation_spec[
        "distribution_evaluation"
    ][
        "simulation_distribution"
    ]
    == "Poisson"
)

assert (
    evaluation_spec[
        "final_test"
    ][
        "matches"
    ]
    == 78
)

assert (
    evaluation_spec[
        "final_test"
    ][
        "actual_goals"
    ]
    == 62
)


print(
    "PHASE 8 EVALUATION ARTIFACT SAVED"
)

print("=" * 75)

print(
    "Saved:",
    EVALUATION_SPEC_PATH
)

print(
    "\nSelected model:",
    evaluation_spec[
        "selected_model"
    ]["name"]
)

print(
    "Final-test deviance:",
    round(
        evaluation_spec[
            "selected_model"
        ][
            "final_test_poisson_deviance"
        ],
        6,
    )
)

print(
    "Final-test bias:",
    round(
        evaluation_spec[
            "selected_model"
        ][
            "final_test_mean_bias"
        ],
        6,
    )
)

print(
    "\nOOF Pearson dispersion:",
    round(
        evaluation_spec[
            "distribution_evaluation"
        ][
            "oof_pearson_dispersion"
        ],
        6,
    )
)

print(
    "Simulation distribution:",
    evaluation_spec[
        "distribution_evaluation"
    ][
        "simulation_distribution"
    ]
)

print(
    "Model changed after test:",
    evaluation_spec[
        "selected_model"
    ][
        "changed_after_final_test"
    ]
)

print(
    "\nPhase 8 persistence: PASSED"
)

PHASE 8 EVALUATION ARTIFACT SAVED
Saved: ..\artifacts\evaluation_spec.json

Selected model: 60% XGBoost + 40% CatBoost ensemble
Final-test deviance: 0.803222
Final-test bias: 0.011189

OOF Pearson dispersion: 0.97915
Simulation distribution: Poisson
Model changed after test: False

Phase 8 persistence: PASSED
